<a href="https://colab.research.google.com/github/malyadri6761/RAG/blob/master/rag_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

**Basic RAG Implementation**

---

```

      Documents
          ↓
      Load
          ↓
      Split into chunks
          ↓
      Create embeddings
          ↓
      Store in vector database
          ↓
      User query
          ↓
      Similarity search
          ↓
      Retrieve relevant chunks
          ↓
      LLM
          ↓
      Answer

      We will not use LangChain initially. Building the components ourselves will make the RAG concepts much clearer.

```

---

**Install Packages**

---

In [ ]:
!pip install sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.9 MB/s eta 0:00:00


---

**2.Creating Documents**

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
path = "/content/drive/MyDrive/RAG/RAG_01"


In [ ]:
ai = """Artificial Intelligence is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence.

Machine learning is a subset of artificial intelligence. It allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn complex patterns from large amounts of data."""


In [ ]:
ml = """Supervised learning is a machine learning approach where a model learns from labeled training data.

Classification is a supervised learning task where the goal is to predict a discrete class or category.

Regression is a supervised learning task where the goal is to predict a continuous numerical value.

Unsupervised learning works with data that does not have labeled outputs. Clustering is one example of unsupervised learning."""

In [39]:
ai_path = os.path.join(path,"ai.txt")

with open(ai_path,"w") as f:
  f.write(ai)

ml_path = os.path.join(path,"ml.txt")

with open(ml_path,"w") as f:
  f.write(ml)


In [40]:
files = os.listdir(path)
files

['ai.txt', 'ml.txt']

---

**3. Load Documents**

---

In [41]:
from pathlib import Path
data_path = Path("/content/drive/MyDrive/RAG/RAG_01")
docs = []

for file in data_path.glob("*.txt"):
  text = file.read_text()

  docs.append({
      'text':text,
      'source':file.name
  })

for doc in docs:
  print("Source : ",doc['source'])
  print(doc['text'])
  print("---"*50)

Source :  ai.txt
Artificial Intelligence is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence.

Machine learning is a subset of artificial intelligence. It allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn complex patterns from large amounts of data.
------------------------------------------------------------------------------------------------------------------------------------------------------
Source :  ml.txt
Supervised learning is a machine learning approach where a model learns from labeled training data.

Classification is a supervised learning task where the goal is to predict a discrete class or category.

Regression is a supervised learning task where the goal is to predict a continuous numerical value.

Unsupervis

---

**4. Chunk the Documents**

---

```
  Document
   ↓
  Chunks
   ↓
  Embeddings

```

In [42]:
def create_chunks(text,chunk_size=10):
  words = text.split()

  chunks = []
  for i in range(0,len(words),chunk_size):
    chunk = " ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks

In [43]:
all_chunks = []
for document in docs:
  chunks = create_chunks(document['text'])

  for chunk in chunks:
    all_chunks.append({
        'text':chunk,
        'source':document['source']
    })
for i,chunk in enumerate(all_chunks):
  print(f"chunk : {i+1}")
  print("source : ",chunk['source'])
  print(chunk['text'])
  print("-"*50)

chunk : 1
source :  ai.txt
Artificial Intelligence is a field of computer science that focuses
--------------------------------------------------
chunk : 2
source :  ai.txt
on creating systems capable of performing tasks that normally require
--------------------------------------------------
chunk : 3
source :  ai.txt
human intelligence. Machine learning is a subset of artificial intelligence.
--------------------------------------------------
chunk : 4
source :  ai.txt
It allows computers to learn patterns from data and make
--------------------------------------------------
chunk : 5
source :  ai.txt
predictions or decisions without being explicitly programmed for every task.
--------------------------------------------------
chunk : 6
source :  ai.txt
Deep learning is a subset of machine learning that uses
--------------------------------------------------
chunk : 7
source :  ai.txt
neural networks with multiple layers to learn complex patterns from
--------------------------------

---

**5. Creating Embeddings**

---

In [44]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

text = [chunk['text'] for chunk in all_chunks]
embeddings = model.encode(text)

print("No.of chunks : ",len(text))
print("Embeddings : ",embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

No.of chunks :  15
Embeddings :  (15, 384)


---

**6. Vector Database**

---

```
  FAISS - Performs efficient similarity search over vectors
```

In [45]:
import faiss
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

query = "What is supervised learning?"
query_embedding = model.encode([query])
distances, indices = index.search(query_embedding, k=3)

print("\nQuery : ",query)
print("\nRetrieved documents : ")
for i, dist in zip(indices[0], distances[0]):
  print("\n Score : ", dist)
  print(all_chunks[i]["text"])
  print("-"*50)


Query :  What is supervised learning?

Retrieved documents : 

 Score :  0.29673257
Supervised learning is a machine learning approach where a model
--------------------------------------------------

 Score :  0.4185029
learns from labeled training data. Classification is a supervised learning
--------------------------------------------------

 Score :  0.9468646
is one example of unsupervised learning.
--------------------------------------------------


---

**7. Understand the retrieval process**

---
```
                      Query
                      ↓
          "What is supervised learning?"
                      ↓
                Embedding Model
                      ↓
                 Query Vector
                      ↓
                 FAISS Search
                      ↓
             ┌────────┼────────┐
             ↓        ↓        ↓
          Chunk 1   Chunk 2   Chunk 3
```

                          ┌──────────────┐
                          │  Documents   │
                          └──────┬───────┘
                                ↓
                          ┌──────────────┐
                          │   Chunking   │
                          └──────┬───────┘
                                ↓
                          ┌──────────────┐
                          │  Embeddings  │
                          └──────┬───────┘
                                ↓
                          ┌──────────────┐
                          │   FAISS DB   │
                          └──────┬───────┘
                                │
                                │
      User Query ────────────────┘
            ↓
        Embedding
            ↓
      Similarity Search
            ↓
      Top-K Chunks
            ↓
      ┌──────────────────────┐
      │ Context + User Query │
      └──────────┬───────────┘
                  ↓
                LLM
                  ↓
              Answer